# 🌿 GreenNetwork Training - Energy-Aware SDN Routing

## GPU-Optimized Training on Google Colab

This notebook trains a DQN agent with adaptive clustering for energy-efficient SDN routing.

**Features:**
- ✅ Auto-detect CUDA/CPU
- ✅ Google Drive integration for model saving
- ✅ Live training visualization
- ✅ Checkpoint management
- ✅ GPU memory monitoring

## 📦 Step 1: Install Dependencies & Check GPU

In [ ]:
# Install required packages
!pip install -q networkx matplotlib pandas plotly tqdm

# Verify PyTorch and GPU
import torch
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"\nCUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\n✅ GPU is ready for training!")
else:
    print("\n⚠️  No GPU detected. Training will use CPU (slower).")
    print("   To enable GPU: Runtime > Change runtime type > GPU")

## 💾 Step 2: Mount Google Drive (Optional but Recommended)

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create save directory
save_dir = '/content/drive/MyDrive/GreenNetwork_Results'
os.makedirs(save_dir, exist_ok=True)
print(f"✅ Models will be saved to: {save_dir}")

## 📥 Step 3: Clone Repository

In [ ]:
# Clone the repository
!git clone https://github.com/andy9310/Research-GreenNetwork.git

# Navigate to training directory
%cd Research-GreenNetwork/train

# List files
!ls -la

## 🔧 Step 4: GPU Utility Functions

In [ ]:
import torch

def clear_gpu_memory():
    """Clear GPU cache"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("✅ GPU cache cleared")

def get_gpu_memory_usage():
    """Get current GPU memory usage"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1e9
        reserved = torch.cuda.memory_reserved(0) / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU Memory: {allocated:.2f}GB / {total:.2f}GB ({allocated/total*100:.1f}%)")
        return allocated, reserved
    return 0, 0

def monitor_gpu():
    """Monitor GPU status"""
    if torch.cuda.is_available():
        print(f"\n📊 GPU Status:")
        get_gpu_memory_usage()
        print(f"   Device: {torch.cuda.get_device_name(0)}")
    else:
        print("\n⚠️  No GPU available")

# Test GPU utilities
monitor_gpu()

## 🎯 Step 5: Configure Training Parameters

In [ ]:
import json

# Load default config
with open('config.json', 'r') as f:
    config = json.load(f)

# Override settings for Colab
config['config']['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
config['config']['episodes'] = 200  # Adjust based on your needs
config['config']['batch_size'] = 512 if torch.cuda.is_available() else 256  # Larger batch on GPU

# Save modified config
with open('config_colab.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✅ Configuration updated for Colab")
print(f"   Device: {config['config']['device']}")
print(f"   Episodes: {config['config']['episodes']}")
print(f"   Batch Size: {config['config']['batch_size']}")
print(f"   Network: {config['config']['num_nodes']} nodes, {config['config']['num_edges']} edges")

## 🚀 Step 6: Run Training

In [ ]:
from train import run_training
import time

# Clear GPU memory before training
clear_gpu_memory()

# Start training
print("\n" + "="*80)
print("🎬 STARTING TRAINING".center(80))
print("="*80 + "\n")

start_time = time.time()

try:
    results = run_training('config_colab.json', traffic_mode='low')
    
    training_time = time.time() - start_time
    
    print("\n" + "="*80)
    print("✅ TRAINING COMPLETED".center(80))
    print("="*80)
    print(f"\n📊 Final Results:")
    print(f"   Total Time: {training_time/60:.1f} minutes")
    print(f"   Best Reward: {results['best_reward']:.2f}")
    print(f"   Avg Reward: {results['avg_reward']:.2f}")
    print(f"   Energy Saving: {results['final_energy_saving']*100:.1f}%")
    print(f"   Latency: {results['final_latency']:.2f}ms")
    print(f"   SLA Violations: {results['final_sla_violations']:.1f}%")
    print(f"   Model saved: {results['final_path']}")
    
except KeyboardInterrupt:
    print("\n⚠️  Training interrupted by user")
except Exception as e:
    print(f"\n❌ Training failed: {e}")
    import traceback
    traceback.print_exc()

# Monitor GPU after training
monitor_gpu()

## 📊 Step 7: Visualize Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load training metrics
metrics_df = pd.read_csv('training_results/training_metrics.csv')

# Display summary statistics
print("📈 Training Summary (Last 50 Episodes):")
print(metrics_df.tail(50).describe())

# Create interactive plots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Reward', 'Energy Saving (%)', 'Latency (ms)', 'SLA Violations (%)'),
    vertical_spacing=0.12
)

# Reward
fig.add_trace(
    go.Scatter(x=metrics_df['episode'], y=metrics_df['reward'], 
              mode='lines', name='Reward', line=dict(color='blue', width=2)),
    row=1, col=1
)

# Energy Saving
fig.add_trace(
    go.Scatter(x=metrics_df['episode'], y=metrics_df['energy_saving']*100, 
              mode='lines', name='Energy Saving', line=dict(color='green', width=2)),
    row=1, col=2
)
fig.add_hline(y=70, line_dash="dash", line_color="red", opacity=0.5, row=1, col=2)

# Latency
fig.add_trace(
    go.Scatter(x=metrics_df['episode'], y=metrics_df['latency'], 
              mode='lines', name='Latency', line=dict(color='magenta', width=2)),
    row=2, col=1
)

# SLA Violations
fig.add_trace(
    go.Scatter(x=metrics_df['episode'], y=metrics_df['sla_violations'], 
              mode='lines', name='SLA Violations', line=dict(color='orange', width=2)),
    row=2, col=2
)
fig.add_hline(y=10, line_dash="dash", line_color="red", opacity=0.5, row=2, col=2)

fig.update_layout(height=700, showlegend=False, title_text="Training Progress")
fig.show()

## 💾 Step 8: Save Results to Google Drive

In [ ]:
import shutil
from datetime import datetime

# Create timestamped folder
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
result_folder = f'{save_dir}/training_{timestamp}'
os.makedirs(result_folder, exist_ok=True)

# Copy all results
shutil.copytree('training_results', f'{result_folder}/training_results', dirs_exist_ok=True)

# Copy model files
import glob
for model_file in glob.glob('*.pth'):
    shutil.copy(model_file, result_folder)

# Copy config
shutil.copy('config_colab.json', result_folder)

print(f"✅ All results saved to: {result_folder}")
print(f"\nSaved files:")
!ls -lh {result_folder}

## 🧪 Step 9: Run Comparative Experiments (Optional)

In [ ]:
from experiment.run_experiments import ExperimentRunner

# Clear GPU memory
clear_gpu_memory()

# Initialize experiment runner
runner = ExperimentRunner(output_dir=f'{save_dir}/experiments_{timestamp}')

# Run experiments on small topologies (suitable for Colab)
print("\n🧪 Running comparative experiments...\n")

runner.run_all_experiments(
    topologies=[20, 100, 500],  # Start with smaller topologies
    methods=['dqn_clustering', 'energy_aware'],  # Skip RL-ER (slower)
    episodes_per_method={
        'dqn_clustering': 500,  # Reduced for Colab
        'energy_aware': 50
    }
)

print("\n✅ Experiments completed!")
print(f"   Results saved to: {save_dir}/experiments_{timestamp}")

## 📈 Step 10: Analyze Experiment Results

In [ ]:
# Load comparison results
comparison_df = pd.read_csv(f'{save_dir}/experiments_{timestamp}/comparison_table.csv')

print("\n📊 Comparative Results:\n")
print(comparison_df.to_string(index=False))

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

methods = comparison_df['Method'].unique()
topologies = comparison_df['Topology'].unique()

# Energy Saving
for method in methods:
    data = comparison_df[comparison_df['Method'] == method]
    axes[0, 0].plot(data['Topology'], data['Energy_Saving_%'], marker='o', label=method)
axes[0, 0].set_title('Energy Saving vs Topology Size')
axes[0, 0].set_xlabel('Number of Links')
axes[0, 0].set_ylabel('Energy Saving (%)')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Latency
for method in methods:
    data = comparison_df[comparison_df['Method'] == method]
    axes[0, 1].plot(data['Topology'], data['Latency_ms'], marker='o', label=method)
axes[0, 1].set_title('Latency vs Topology Size')
axes[0, 1].set_xlabel('Number of Links')
axes[0, 1].set_ylabel('Latency (ms)')
axes[0, 1].legend()
axes[0, 1].grid(True)

# SLA Violations
for method in methods:
    data = comparison_df[comparison_df['Method'] == method]
    axes[1, 0].plot(data['Topology'], data['SLA_Violations_%'], marker='o', label=method)
axes[1, 0].set_title('SLA Violations vs Topology Size')
axes[1, 0].set_xlabel('Number of Links')
axes[1, 0].set_ylabel('SLA Violations (%)')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Computation Time
for method in methods:
    data = comparison_df[comparison_df['Method'] == method]
    axes[1, 1].plot(data['Topology'], data['Comp_Time_s'], marker='o', label=method)
axes[1, 1].set_title('Computation Time vs Topology Size')
axes[1, 1].set_xlabel('Number of Links')
axes[1, 1].set_ylabel('Computation Time (s)')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig(f'{save_dir}/experiments_{timestamp}/comparison_plots.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Comparison plots saved to: {save_dir}/experiments_{timestamp}/comparison_plots.png")

## 🧹 Step 11: Cleanup & Final GPU Check

In [ ]:
# Clear GPU memory
clear_gpu_memory()

# Final GPU status
monitor_gpu()

print("\n✅ Training session complete!")
print(f"\n📁 All results are saved in Google Drive:")
print(f"   {save_dir}")